## 2. Simulate a small company

a) Connect python to gemini, very important that you place the api key in .env and gitignore it

b) Use gemini to simulate 20 data points in json format containing the following fields: first_name, last_name, phone_number, email, department, salary, title. See if you can prompt to direct the LLM output to have swedish names, phone numbers in swedish format (+46 731 29 52), departments (IT, HR, marketing, sales), reasonable salary (you might need to check some swedish statistics on salaries) and corresponding titles within these departments.

In [16]:
from dotenv import load_dotenv
import os
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
response = client.models.generate_content(model = "gemini-2.5-flash", contents="jag skulle vilja ha 20 datapunkter med följade fält, first_name, last_name, phone_number, email, deparment, salary, title, reglerna är att telefonnumret måste vara i svenskformat alltså +46 format, använd dig av avdelningarna IT, HR, marketing och sales. Ange rimliga löner och titlar för varje avdelning. Make sure the output is a single, valid JSON object and very important that you remove the ```json and ``` in the end "





)

print(response.text)

{
  "employees": [
    {
      "first_name": "Erik",
      "last_name": "Svensson",
      "phone_number": "+46701234567",
      "email": "erik.svensson@example.com",
      "department": "IT",
      "salary": 48000,
      "title": "Software Developer"
    },
    {
      "first_name": "Anna",
      "last_name": "Lindberg",
      "phone_number": "+46702345678",
      "email": "anna.lindberg@example.com",
      "department": "HR",
      "salary": 39000,
      "title": "HR Specialist"
    },
    {
      "first_name": "Johan",
      "last_name": "Karlsson",
      "phone_number": "+46703456789",
      "email": "johan.karlsson@example.com",
      "department": "Marketing",
      "salary": 37000,
      "title": "Digital Marketing Specialist"
    },
    {
      "first_name": "Maria",
      "last_name": "Andersson",
      "phone_number": "+46704567890",
      "email": "maria.andersson@example.com",
      "department": "Sales",
      "salary": 35000,
      "title": "Sales Representative"
    },
  

c) Now use pydantic to validate this json and put in proper schema that the fields should follow. You might need to do some processing such as removing backticks and maybe loading json data into a list with json.loads(). Also make sure that only correctly validated data should be stored.

In [17]:
from pydantic import BaseModel, Field, EmailStr
import json 

raw_json_string = response.text.replace("```json", "").replace("```", "").strip()
json_data = json.loads(raw_json_string)

class employee(BaseModel):
    first_name: str    
    last_name: str 
    phone_number: str = Field(pattern=r"^\+46\s?\d{2}\s?\d{3}\s?\d{2}\s?\d{2}$") 
    email: EmailStr 
    department: str
    salary: float
    title: str

class Company(BaseModel):
    employees: list[employee]

company = Company.model_validate(json_data)
company

Company(employees=[employee(first_name='Erik', last_name='Svensson', phone_number='+46701234567', email='erik.svensson@example.com', department='IT', salary=48000.0, title='Software Developer'), employee(first_name='Anna', last_name='Lindberg', phone_number='+46702345678', email='anna.lindberg@example.com', department='HR', salary=39000.0, title='HR Specialist'), employee(first_name='Johan', last_name='Karlsson', phone_number='+46703456789', email='johan.karlsson@example.com', department='Marketing', salary=37000.0, title='Digital Marketing Specialist'), employee(first_name='Maria', last_name='Andersson', phone_number='+46704567890', email='maria.andersson@example.com', department='Sales', salary=35000.0, title='Sales Representative'), employee(first_name='Daniel', last_name='Olsson', phone_number='+46705678901', email='daniel.olsson@example.com', department='IT', salary=43000.0, title='Systems Administrator'), employee(first_name='Sofia', last_name='Berg', phone_number='+46706789012',

d) Write this json data to a folder called output_data.